# MySQL JDBC Incremental Ingestion - Claims Database

This notebook implements **incremental data ingestion** from MySQL using JDBC with watermark-based change tracking.

## Approach

* **Initial Load**: Full table load on first run
* **Incremental Load**: Use watermark columns (timestamps/dates) to fetch only new/updated records
* **Merge Strategy**: Delta Lake MERGE for upsert operations
* **Watermark Tracking**: Store last processed values in control table

## Tables

* `policy` - Insurance policies (Primary Key: policy_no)
* `claim` - Claims data (Primary Key: claim_no)
* `customer` - Customer information (Primary Key: customer_id)

## Configuration

All settings are configured via **query parameters** at the top of this notebook.

**Source**: MySQL localhost (127.0.0.1:3306)
**Target**: Unity Catalog (main.claims_mysql)

In [0]:
# Load configuration from query parameters
config = {
    'mysql_host': dbutils.widgets.get('mysql_host'),
    'mysql_port': dbutils.widgets.get('mysql_port'),
    'mysql_database': dbutils.widgets.get('mysql_database'),
    'mysql_user': dbutils.widgets.get('mysql_user'),
    'mysql_password': dbutils.widgets.get('mysql_password'),
    'target_catalog': dbutils.widgets.get('target_catalog'),
    'target_schema': dbutils.widgets.get('target_schema')
}

# Build JDBC URL
jdbc_url = f"jdbc:mysql://{config['mysql_host']}:{config['mysql_port']}/{config['mysql_database']}"

# Display configuration (mask password)
print("MySQL JDBC Configuration:")
print(f"  Host: {config['mysql_host']}")
print(f"  Port: {config['mysql_port']}")
print(f"  Database: {config['mysql_database']}")
print(f"  User: {config['mysql_user']}")
print(f"  Password: {'*' * 8}")
print(f"  JDBC URL: {jdbc_url}")
print(f"\nTarget:")
print(f"  Catalog: {config['target_catalog']}")
print(f"  Schema: {config['target_schema']}")
print(f"  Full Path: {config['target_catalog']}.{config['target_schema']}")

print("\n✓ Configuration loaded")

In [0]:
# JDBC connection properties for MySQL
jdbc_properties = {
    "user": config['mysql_user'],
    "password": config['mysql_password'],
    "driver": "com.mysql.cj.jdbc.Driver",
    "serverTimezone": "UTC",
    "useSSL": "false",
    "allowPublicKeyRetrieval": "true"
}

print("✓ JDBC properties configured")
print("\nDriver: MySQL Connector/J (included in Databricks Runtime)")

In [0]:
# Create Unity Catalog schema if it doesn't exist

target_location = f"{config['target_catalog']}.{config['target_schema']}"

try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_location}")
    print(f"✓ Schema {target_location} ready")
    
    # Verify schema exists
    spark.sql(f"USE {target_location}")
    print(f"✓ Using schema {target_location}")
    
except Exception as e:
    print(f"❌ Error creating schema: {e}")
    raise

In [0]:
# Test connectivity to MySQL

try:
    # Simple test query
    test_df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", "(SELECT 1 as test, DATABASE() as current_db, VERSION() as version) as test") \
        .option("user", config['mysql_user']) \
        .option("password", config['mysql_password']) \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .load()
    
    result = test_df.collect()[0]
    print("✓ MySQL connection successful!")
    print(f"  Database: {result['current_db']}")
    print(f"  Version: {result['version']}")
    
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Verify MySQL is running on localhost")
    print("  2. Check credentials are correct")
    print("  3. Ensure databricks_cdc user has SELECT privileges")
    print(f"  4. Test connection: mysql -h {config['mysql_host']} -u {config['mysql_user']} -p")
    raise

In [0]:
# Define tables with their primary keys and watermark columns
# Watermark columns are used to track incremental changes

tables_config = [
    {
        "name": "policy",
        "primary_key": "policy_no",
        "watermark_column": "pol_issue_date",  # Using issue date as watermark
        "watermark_type": "date",
        "description": "Insurance policy information"
    },
    {
        "name": "claim",
        "primary_key": "claim_no",
        "watermark_column": "claim_date",  # Using claim date as watermark
        "watermark_type": "string",  # Note: claim_date is VARCHAR in source
        "description": "Claims data"
    },
    {
        "name": "customer",
        "primary_key": "customer_id",
        "watermark_column": None,  # No timestamp column, will do full load each time
        "watermark_type": None,
        "description": "Customer demographic information"
    }
]

print("Table Configuration:")
print("=" * 80)
for table in tables_config:
    watermark_info = f"{table['watermark_column']} ({table['watermark_type']})" if table['watermark_column'] else "None (full load)"
    print(f"\n{table['name']}:")
    print(f"  Primary Key: {table['primary_key']}")
    print(f"  Watermark: {watermark_info}")
    print(f"  Description: {table['description']}")

print("\n" + "=" * 80)
print("✓ Table metadata defined")

In [0]:
# Create a control table to track last watermark values for incremental loads

control_table = f"{target_location}.ingestion_watermarks"

try:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {control_table} (
            table_name STRING NOT NULL,
            last_watermark_value STRING,
            last_load_timestamp TIMESTAMP,
            row_count BIGINT,
            load_type STRING,
            PRIMARY KEY(table_name)
        )
        USING DELTA
        COMMENT 'Tracks watermark values for incremental ingestion'
    """)
    
    print(f"✓ Control table ready: {control_table}")
    
    # Show current watermarks if any
    watermarks_df = spark.sql(f"SELECT * FROM {control_table}")
    if watermarks_df.count() > 0:
        print("\nCurrent Watermarks:")
        display(watermarks_df)
    else:
        print("\n(No watermarks yet - first run will be full load)")
        
except Exception as e:
    print(f"❌ Error creating control table: {e}")
    raise

## Core Ingestion Logic

The `load_table()` function handles both initial full load and incremental updates:

* **First Run**: Full table load
* **Subsequent Runs**: Incremental load based on watermark column
* **Upsert**: Delta MERGE to handle inserts and updates
* **Watermark**: Automatically tracked in control table

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, max as spark_max
from delta.tables import DeltaTable

def load_table(table_name, primary_key, watermark_column=None):
    """
    Load data from MySQL table using incremental approach
    
    Args:
        table_name: Source MySQL table name
        primary_key: Primary key column for MERGE
        watermark_column: Column to use for incremental tracking (None = full load)
    """
    target_table = f"{target_location}.{table_name}"
    
    print(f"\n{'='*80}")
    print(f"Loading table: {table_name}")
    print(f"{'='*80}")
    
    # Get last watermark value
    last_watermark = None
    load_type = "FULL"
    
    if watermark_column:
        try:
            watermark_df = spark.sql(f"""
                SELECT last_watermark_value 
                FROM {control_table} 
                WHERE table_name = '{table_name}'
            """)
            
            if watermark_df.count() > 0:
                last_watermark = watermark_df.collect()[0]['last_watermark_value']
                load_type = "INCREMENTAL"
                print(f"  Last watermark: {last_watermark}")
            else:
                print(f"  No watermark found - performing full load")
        except:
            print(f"  No watermark found - performing full load")
    else:
        print(f"  No watermark column - performing full load")
    
    # Build query
    if watermark_column and last_watermark:
        query = f"(SELECT * FROM {table_name} WHERE {watermark_column} > '{last_watermark}') as incremental_data"
        print(f"  Query: SELECT * FROM {table_name} WHERE {watermark_column} > '{last_watermark}'")
    else:
        query = table_name
        print(f"  Query: SELECT * FROM {table_name}")
    
    # Read from MySQL
    print(f"  Reading from MySQL...")
    source_df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", query) \
        .option("user", config['mysql_user']) \
        .option("password", config['mysql_password']) \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .load()
    
    row_count = source_df.count()
    print(f"  Rows fetched: {row_count:,}")
    
    if row_count == 0:
        print(f"  ✓ No new records to process")
        return
    
    # Check if target table exists
    table_exists = spark.catalog.tableExists(target_table)
    
    if not table_exists:
        # First load - create table
        print(f"  Creating new table: {target_table}")
        source_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)
        print(f"  ✓ Table created with {row_count:,} rows")
    else:
        # Table exists - perform MERGE
        print(f"  Performing MERGE into {target_table}")
        
        delta_table = DeltaTable.forName(spark, target_table)
        
        # Create merge condition on primary key
        merge_condition = f"target.{primary_key} = source.{primary_key}"
        
        # Perform MERGE
        delta_table.alias("target").merge(
            source_df.alias("source"),
            merge_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        
        print(f"  ✓ MERGE completed")
    
    # Update watermark
    if watermark_column:
        new_watermark = source_df.agg(spark_max(col(watermark_column))).collect()[0][0]
        if new_watermark:
            new_watermark_str = str(new_watermark)
            print(f"  New watermark: {new_watermark_str}")
            
            # Upsert watermark
            spark.sql(f"""
                MERGE INTO {control_table} target
                USING (SELECT 
                    '{table_name}' as table_name,
                    '{new_watermark_str}' as last_watermark_value,
                    current_timestamp() as last_load_timestamp,
                    {row_count} as row_count,
                    '{load_type}' as load_type
                ) source
                ON target.table_name = source.table_name
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            print(f"  ✓ Watermark updated")
    else:
        # No watermark - just record the load
        spark.sql(f"""
            MERGE INTO {control_table} target
            USING (SELECT 
                '{table_name}' as table_name,
                NULL as last_watermark_value,
                current_timestamp() as last_load_timestamp,
                {row_count} as row_count,
                'FULL' as load_type
            ) source
            ON target.table_name = source.table_name
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
    
    # Get final row count
    final_count = spark.table(target_table).count()
    print(f"  Total rows in table: {final_count:,}")
    print(f"  ✓ Load complete ({load_type})\n")

print("✓ Incremental load function defined")

## Execute Data Ingestion

Run the ingestion for all three tables. On first run, this will do a full load. Subsequent runs will be incremental based on watermark columns.

In [0]:
# Load all tables using the configuration defined earlier

import time
start_time = time.time()

print("\n" + "#" * 80)
print("# STARTING DATA INGESTION")
print("#" * 80)

# Load each table
for table in tables_config:
    try:
        load_table(
            table_name=table['name'],
            primary_key=table['primary_key'],
            watermark_column=table['watermark_column']
        )
    except Exception as e:
        print(f"\n❌ ERROR loading {table['name']}: {e}")
        raise

elapsed = time.time() - start_time
print("\n" + "#" * 80)
print(f"# INGESTION COMPLETE - Elapsed time: {elapsed:.2f} seconds")
print("#" * 80)

# Show watermark status
print("\n\nWatermark Status:")
display(spark.table(control_table))

## Data Verification

Verify that data was ingested correctly.

In [0]:
# Verify row counts for all tables

print("Table Row Counts:")
print("=" * 80)

for table in tables_config:
    table_name = f"{target_location}.{table['name']}"
    try:
        count = spark.table(table_name).count()
        print(f"  {table['name']}: {count:,} rows")
    except Exception as e:
        print(f"  {table['name']}: Table not found or error - {e}")

print("=" * 80)

In [0]:
%sql
-- View sample data from policy table
SELECT 
  policy_no,
  cust_id,
  policytype,
  pol_issue_date,
  pol_eff_date,
  pol_expiry_date,
  make,
  model,
  model_year,
  sum_insured,
  premium
FROM main.claims_mysql.policy
LIMIT 10

In [0]:
%sql
-- View sample data from claim table
SELECT 
  claim_no,
  policy_no,
  claim_date,
  total,
  collision_type,
  incident_type,
  incident_severity,
  suspicious_activity
FROM main.claims_mysql.claim
LIMIT 10

In [0]:
%sql
-- View sample data from customer table
SELECT 
  customer_id,
  name,
  date_of_birth,
  borough,
  neighborhood,
  zip_code
FROM main.claims_mysql.customer
LIMIT 10

In [0]:
%sql
-- Join claims with policies to verify relationships
SELECT 
  c.claim_no,
  c.policy_no,
  c.total as claim_amount,
  c.incident_type,
  c.incident_severity,
  p.policytype,
  p.make,
  p.model,
  p.sum_insured,
  p.cust_id
FROM main.claims_mysql.claim c
INNER JOIN main.claims_mysql.policy p 
  ON c.policy_no = p.policy_no
ORDER BY c.total DESC
LIMIT 20

## Testing Incremental Updates

To test that incremental loading works correctly:

### 1. Insert Test Data in MySQL
```sql
-- Run this in your MySQL database
INSERT INTO policy (policy_no, cust_id, policytype, pol_issue_date, pol_eff_date, pol_expiry_date, make, model) 
VALUES ('TEST999', 'CUST999', 'Auto', '2026-04-01', '2026-04-01', '2027-04-01', 'Tesla', 'Model 3');
```

### 2. Re-run Ingestion
Run **[Cell 11: Load All Tables](#cell-d1e30a02-0347-403e-a2f4-f325ab6b6fff)** again

### 3. Verify New Record
```sql
SELECT * FROM main.claims_mysql.policy WHERE policy_no = 'TEST999';
```

### 4. Check Watermark
The watermark should update to the new `pol_issue_date`:
```python
display(spark.table(f"{target_location}.ingestion_watermarks"))
```

### Expected Behavior:
* **First Run**: Full load of all records
* **Second Run**: Only records with `pol_issue_date > last_watermark`
* **MERGE**: New records inserted, existing records updated based on primary key

## Scheduling the Ingestion

You have several options to schedule this notebook to run periodically:

### Option 1: Databricks Workflow (Recommended)

1. Click **Schedule** in the top right of this notebook
2. Create a new job schedule:
   * **Name**: `MySQL Claims Incremental Ingestion`
   * **Schedule**: Every 1 hour (or as needed)
   * **Cluster**: Shared or dedicated cluster
3. Click **Create**

### Option 2: Databricks Jobs UI

1. Go to **Workflows** in the sidebar
2. Click **Create Job**
3. Add this notebook as a task
4. Configure schedule and cluster
5. Add email notifications

### Option 3: REST API

```python
# Example using Databricks REST API
import requests

job_config = {
    "name": "MySQL Claims Incremental Ingestion",
    "tasks": [{
        "task_key": "ingest_mysql",
        "notebook_task": {
            "notebook_path": "/Users/mgmarques3000@gmail.com/MySQL JDBC Incremental Ingestion - Claims Dev",
            "source": "WORKSPACE"
        },
        "new_cluster": {
            "spark_version": "13.3.x-scala2.12",
            "node_type_id": "i3.xlarge",
            "num_workers": 2
        }
    }],
    "schedule": {
        "quartz_cron_expression": "0 0 */1 ? * *",  # Every hour
        "timezone_id": "UTC"
    },
    "email_notifications": {
        "on_failure": ["your-email@example.com"]
    }
}
```

### Recommended Schedule

* **Dev/Test**: Manual runs or every 6 hours
* **Production**: Every 1-4 hours depending on data freshness requirements
* **Near Real-Time**: Every 15-30 minutes (requires more compute)

### Monitoring

Check the `ingestion_watermarks` table to monitor:
* Last successful load time
* Number of rows processed
* Load type (FULL vs INCREMENTAL)

In [0]:
# Helper to create a job schedule for this notebook
# Uncomment and run to schedule via Python

# from databricks.sdk import WorkspaceClient
# from databricks.sdk.service.jobs import Task, NotebookTask, JobCluster
# 
# w = WorkspaceClient()
# 
# job = w.jobs.create(
#     name="MySQL Claims Incremental Ingestion",
#     tasks=[
#         Task(
#             task_key="ingest_mysql_claims",
#             notebook_task=NotebookTask(
#                 notebook_path="/Users/mgmarques3000@gmail.com/MySQL JDBC Incremental Ingestion - Claims Dev",
#                 source="WORKSPACE"
#             ),
#             new_cluster=JobCluster(
#                 spark_version="13.3.x-scala2.12",
#                 node_type_id="i3.xlarge",
#                 num_workers=2
#             )
#         )
#     ],
#     schedule={
#         "quartz_cron_expression": "0 0 */1 ? * *",  # Every hour
#         "timezone_id": "UTC"
#     }
# )
# 
# print(f"Job created: {job.job_id}")
# print(f"Job URL: {w.config.host}/#job/{job.job_id}")

print("✓ Uncomment code above to create scheduled job")
print("\nOr use the 'Schedule' button in the top right of this notebook")

## ✓ Setup Complete!

You now have a working JDBC-based incremental ingestion pipeline from MySQL to Unity Catalog.

### What You Built:

* **Configuration**: Query parameters for easy customization
* **Connection**: JDBC connection to MySQL localhost
* **Watermark Tracking**: Control table tracks last processed values
* **Incremental Logic**: Fetch only new/updated records
* **MERGE/Upsert**: Delta Lake MERGE for efficient updates
* **Monitoring**: Row counts, watermarks, and load statistics

### Tables Ingested:

| Table | Primary Key | Watermark Column | Strategy |
|-------|-------------|------------------|----------|
| policy | policy_no | pol_issue_date | Incremental |
| claim | claim_no | claim_date | Incremental |
| customer | customer_id | None | Full load |

### Key Benefits:

* **Simple**: No special CDC tools required
* **Efficient**: Only fetch changed data
* **Reliable**: Delta Lake ACID guarantees
* **Flexible**: Easy to customize and extend
* **Dev-Friendly**: Works with localhost MySQL

### Next Steps:

1. **Test**: Insert new records in MySQL and re-run ingestion
2. **Schedule**: Set up periodic runs via Workflows
3. **Monitor**: Check watermarks and row counts regularly
4. **Enhance**: Add error handling, alerting, or additional tables
5. **Production**: Update connection to production MySQL endpoint

### Resources:

* **Notebook**: `/Users/mgmarques3000@gmail.com/MySQL JDBC Incremental Ingestion - Claims Dev`
* **Target**: `main.claims_mysql.*`
* **Control Table**: `main.claims_mysql.ingestion_watermarks`

---

**Ready for production?** Update the `mysql_host` parameter to point to your production MySQL server and schedule the job!